# 09 — CBAM Cost Estimation: Default Emission Values

Estimates the total CBAM certificate cost for each exporting country and sector,
based on 2024 EU import volumes and EU Commission default emission values.

**Methodology:**
- Estimated cost (EUR) = import volume (tonnes) x default emission value (tCO2/t) x certificate price (EUR/tCO2)
- Default emission values in `default_2026` already include the applicable markup, as confirmed
  by the source xlsx column header 'including mark-up' (10% for most sectors, 1% for fertilizers).
  No additional markup is applied in this notebook.
- Certificate price: EUR 75.36/tCO2 (first official CBAM price, European Commission, April 2026)
- Trade flow year: 2024, QUANTITY_IN_TONNES indicator only

**Join logic:**
- cbam_defaults is the left (anchor) table. All 119 countries with published defaults
  are retained regardless of whether they have matching 2024 trade flow data.
- Countries with no trade flow match receive import_tonnes = 0 and cbam_cost_eur = 0.
  This represents a zero CBAM exposure, not missing data.
- The 116 trade flow countries with no CBAM default (EU member states, small territories,
  non-CBAM exporters) are excluded. This is expected and correct.
- Sector labels are derived from the cbam_defaults side via an explicit CN code mapping,
  so all rows including zero-trade rows carry a sector label.
- Where a country has multiple production routes per CN code, the worst-case
  (highest default_2026) route is used to avoid double-counting import volume.

**Known limitations:**
- CBAM certificate price is parameterized and should be updated as the market matures.
- The markup in default_2026 is specific to 2026. For future-year analysis,
  use default_2027 (20% markup) or default_2028_onwards (30% markup).

**Output tables written to db/cbam.db:**
- `cbam_cost_by_country_sector` — grain: (country, sector, cn_code)
- `cbam_cost_by_country` — grain: (country), aggregated across all sectors
- `cbam_cost_by_sector` — grain: (sector), aggregated across all countries

In [40]:
# ── Imports ───────────────────────────────────────────────────────────────────
# Standard libraries for data manipulation, database access, and path handling.

import sqlite3
import pandas as pd
from pathlib import Path

In [41]:
# ── Constants ─────────────────────────────────────────────────────────────────
# All key assumptions are defined here as named constants so they can be
# reviewed, challenged, and updated in one place without touching calculation logic.
#
# CERTIFICATE_PRICE_EUR:
#   First official CBAM certificate price published by the European Commission
#   on 7 April 2026. This price tracks the EU ETS and will fluctuate over time.
#   Source: European Commission via
#   https://www.homaio.com/post/eu-ets-definitions-updated-guide-for-2025
#   Update this constant to reflect a different price period if needed.
#
# REFERENCE_YEAR:
#   Trade flow data year. 2024 is the most recent complete year available
#   and the last full year before CBAM's definitive phase began (1 January 2026).
#
# SECTOR_MAP:
#   Maps each CBAM CN code prefix to its sector label. Derived from the defaults
#   table rather than the trade flows material column, so sector labels are
#   available for all 119 countries including those with no trade flow data.
#   Electricity (CN 2716) is included for completeness even though it has
#   negligible trade flow volume in COMEXT.

CERTIFICATE_PRICE_EUR = 75.36   # EUR/tCO2, first official EC CBAM price, April 2026
REFERENCE_YEAR        = 2024    # Trade flow reference year

SECTOR_MAP = {
    '2507': 'Cement',
    '2523': 'Cement',
    '6810': 'Cement',
    '6811': 'Cement',
    '2601': 'Iron and Steel',
    '2716': 'Electricity',
    '2804': 'Hydrogen',
    '2808': 'Fertilizers',
    '2814': 'Fertilizers',
    '2834': 'Fertilizers',
    '3102': 'Fertilizers',
    '3105': 'Fertilizers',
    '72'  : 'Iron and Steel',
    '73'  : 'Iron and Steel',
    '76'  : 'Aluminium',
}

In [42]:
# ── Display formatting ────────────────────────────────────────────────────────
# Suppress scientific notation for float columns in pandas display and output.
# Applies globally for this notebook session.

pd.set_option('display.float_format', '{:,.2f}'.format)

In [43]:
# ── Database connection ───────────────────────────────────────────────────────
# Connect to the SQLite database generated in notebook 08.
# projects/01_country_exposure/ is two levels below the repo root,
# so ../../db/cbam.db points to db/cbam.db at the repo root.

DB_PATH = Path('../../db/cbam.db')

assert DB_PATH.exists(), (
    f'Database not found at {DB_PATH.resolve()}.\n'
    f'Run notebook 08 first to generate cbam.db.'
)

con = sqlite3.connect(DB_PATH)
print(f'Connected to: {DB_PATH.resolve()}')

Connected to: /Users/milcahmaryjoseph/Documents/GitHub/cbam-analysis/db/cbam.db


In [44]:
# ── Load CBAM default emission values ────────────────────────────────────────
# Pull 2026 default emission values. This is the left (anchor) table.
# All 119 countries with a published default will appear in the final output.
#
# Rows where default_2026 IS NULL are excluded. One known case: Chile CN 73061100,
# confirmed as a dash in the legally binding regulation (EUR-Lex page 471/2400).

query_defaults = """
    SELECT
        country,
        cn_code,
        production_route_code,
        production_route,
        direct_emissions,
        indirect_emissions,
        total_emissions,
        default_2026
    FROM cbam_defaults
    WHERE default_2026 IS NOT NULL
"""

df_defaults = pd.read_sql(query_defaults, con)
print(f'CBAM default rows loaded: {len(df_defaults):,}')
print(f'Countries with defaults:  {df_defaults["country"].nunique()}')
print(f'Unique CN codes:          {df_defaults["cn_code"].nunique()}')
df_defaults.head()

CBAM default rows loaded: 10,670
Countries with defaults:  119
Unique CN codes:          262


,country,cn_code,production_route_code,production_route,direct_emissions,indirect_emissions,total_emissions,default_2026
0,Albania,25231000,(A),Grey Clinker / Cement,0.87,0.00,0.87,0.96
1,Albania,25232900,NaN,NaN,0.90,0.03,0.93,1.02
2,Albania,25239000,(A),Grey Clinker / Cement,0.86,0.03,0.89,0.98
3,Albania,28080000,NaN,NaN,2.73,0.04,2.76,2.79
4,Albania,28142000,NaN,NaN,0.65,0.03,0.68,0.69


In [45]:
# ── Assign sector labels from CN codes ───────────────────────────────────────
# Sector labels are derived from the defaults table using SECTOR_MAP so that
# all 119 countries carry a sector label, including those with no trade flow data.
# Matching is done longest-prefix-first to avoid shorter keys (e.g. '73') capturing
# codes that should match a longer key (e.g. '2834').
#
# Any CN code that does not match a known prefix raises an error immediately.

def map_sector(cn_code: str) -> str:
    """Return the CBAM sector name for a given CN code string."""
    for prefix in sorted(SECTOR_MAP.keys(), key=len, reverse=True):
        if cn_code.startswith(prefix):
            return SECTOR_MAP[prefix]
    return None

df_defaults['sector'] = df_defaults['cn_code'].apply(map_sector)

unmapped = df_defaults[df_defaults['sector'].isna()]['cn_code'].unique()
if len(unmapped) > 0:
    raise ValueError(
        f'CN codes not matched by SECTOR_MAP: {unmapped}\n'
        f'Add the relevant prefix to SECTOR_MAP in the constants cell.'
    )

print('Sector assignment complete:')
print(df_defaults['sector'].value_counts())

Sector assignment complete:
sector
Iron and Steel    6251
Fertilizers       2389
Aluminium         1608
Cement             329
Hydrogen            93
Name: count, dtype: int64


In [46]:
# ── Deduplicate defaults to one route per (country, cn_code) ──────────────────
# Some country/CN code combinations have multiple production routes.
# Retain the worst-case (highest default_2026) route per (country, cn_code)
# to avoid double-counting import volume when joining to trade flows.
# The selected production_route_code is carried through to the output.

df_defaults_deduped = (
    df_defaults
    .sort_values('default_2026', ascending=False)
    .drop_duplicates(subset=['country', 'cn_code'], keep='first')
    .reset_index(drop=True)
)

rows_removed = len(df_defaults) - len(df_defaults_deduped)
print(f'Default rows before deduplication: {len(df_defaults):,}')
print(f'Default rows after deduplication:  {len(df_defaults_deduped):,}')
print(f'Rows removed (multi-route):        {rows_removed:,}')

Default rows before deduplication: 10,670
Default rows after deduplication:  10,641
Rows removed (multi-route):        29


In [47]:
# ── Load 2024 EU import trade flows ──────────────────────────────────────────
# Pull import volumes in tonnes for the reference year.
#
# Filtering notes:
#   - indicator = 'QUANTITY_IN_TONNES': weight rows only.
#   - The flow column contains only import rows, so no flow filter is required.
#   - iso2 is pulled here to carry through to the output via the join.
#   - Sector and country name will come from the defaults side after the join.

query_trade = f"""
    SELECT
        country,
        iso2,
        cn_code,
        value AS import_tonnes
    FROM trade_flows
    WHERE year      = {REFERENCE_YEAR}
      AND indicator = 'QUANTITY_IN_TONNES'
"""

df_trade = pd.read_sql(query_trade, con)
print(f'Trade flow rows loaded:  {len(df_trade):,}')
print(f'Partner countries:       {df_trade["country"].nunique()}')
print(f'Zero-value rows:         {(df_trade["import_tonnes"] == 0).sum():,}')
df_trade.head()

Trade flow rows loaded:  15,734
Partner countries:       229
Zero-value rows:         290


,country,iso2,cn_code,import_tonnes
0,Afghanistan,AF,25070080,0.10
1,Afghanistan,AF,7210,0.01
2,Afghanistan,AF,73049000,0.35
3,Afghanistan,AF,7308,24.73
4,Afghanistan,AF,7309,0.55


In [48]:
# ── Join trade flows to CBAM defaults (left join from defaults) ───────────────
# Left join with defaults as the anchor ensures all 119 CBAM countries are retained.
# Countries with no matching trade flow rows will have null import_tonnes and iso2,
# which are filled to 0 and None respectively in the next cell.
#
# The 116 trade flow countries not in cbam_defaults are EU member states,
# small territories, and non-CBAM exporters. They are correctly excluded.

df_merged = df_defaults_deduped.merge(
    df_trade[['country', 'iso2', 'cn_code', 'import_tonnes']],
    on=['country', 'cn_code'],
    how='left'
)

# Fill nulls introduced by unmatched trade flow rows
df_merged['import_tonnes'] = df_merged['import_tonnes'].fillna(0)

# iso2 may be null for countries with no trade flow match.
# Fill from country_crosswalk to ensure all rows have an iso2 for mapping.
missing_iso2 = df_merged[df_merged['iso2'].isna()]['country'].unique()
if len(missing_iso2) > 0:
    df_crosswalk = pd.read_sql('SELECT country, iso2 FROM country_crosswalk', con)
    df_merged = df_merged.merge(df_crosswalk, on='country', how='left', suffixes=('', '_cw'))
    df_merged['iso2'] = df_merged['iso2'].fillna(df_merged['iso2_cw'])
    df_merged = df_merged.drop(columns='iso2_cw')
    print(f'iso2 filled from crosswalk for {len(missing_iso2)} countries: {list(missing_iso2)}')

print(f'Rows after join:         {len(df_merged):,}')
print(f'Countries retained:      {df_merged["country"].nunique()}')
print(f'Zero import_tonnes rows: {(df_merged["import_tonnes"] == 0).sum():,}')

iso2 filled from crosswalk for 117 countries: ['South Africa', 'Iran', 'India', 'Jordan', 'Niger', 'Philippines', 'Belarus', 'Albania', 'Saudi Arabia', 'Kuwait', 'Indonesia', 'New Zealand', 'Peru', 'Papua New Guinea', 'United Arab Emirates', 'Pakistan', 'Morocco', 'Malaysia', 'Australia', 'Nicaragua', 'Libya', 'Kazakhstan', 'Colombia', 'Kenya', 'Turkmenistan', 'Tunisia', 'Sierra Leone', 'Vietnam', 'Mexico', 'Azerbaijan', 'Brunei Darussalam', 'Sri Lanka', 'Sudan', 'Suriname', 'Bahrain', 'Congo', 'Nigeria', 'Laos', 'Costa Rica', 'Cameroon', 'Iraq', 'Angola', 'Yemen', 'Oman', 'Venezuela', 'Bosnia and Herzegovina', 'Argentina', 'Bolivia', 'Jamaica', 'Chile', 'Kyrgyzstan', 'Guatemala', 'Uruguay', 'Brazil', 'Eritrea', 'Equatorial Guinea', 'El Salvador', 'Uzbekistan', 'Ecuador', 'Dominican Republic', 'Curacao', 'Cuba', 'Algeria', 'Paraguay', 'North Korea', 'Qatar', 'Russia', 'Syria', 'Trinidad and Tobago', 'Zambia', 'Thailand', 'Bangladesh', 'Tajikistan', 'Taiwan', 'Turkey', 'Mozambique', 'Ca

In [49]:
# ── Core CBAM cost calculation ────────────────────────────────────────────────
# Estimated CBAM cost (EUR) = import volume (tonnes)
#                             x default emission value (tCO2/t, markup already included)
#                             x certificate price (EUR/tCO2)
#
# embedded_co2_tco2 is stored as an intermediate column for interpretability
# and for downstream analysis. Rows with zero import volume produce zero cost.

df_merged['embedded_co2_tco2'] = df_merged['import_tonnes'] * df_merged['default_2026']
df_merged['cbam_cost_eur']     = df_merged['embedded_co2_tco2'] * CERTIFICATE_PRICE_EUR

print('Total estimated CBAM cost (all countries, all sectors):')
print(f'  EUR {df_merged["cbam_cost_eur"].sum():>18,.0f}')
print(f'  {df_merged["embedded_co2_tco2"].sum() / 1e6:>10.2f} MtCO2 embedded')
print(f'  Certificate price used: EUR {CERTIFICATE_PRICE_EUR}/tCO2')

Total estimated CBAM cost (all countries, all sectors):
  EUR     15,549,076,127
      206.33 MtCO2 embedded
  Certificate price used: EUR 75.36/tCO2


In [50]:
# ── Build output table 1: cost by country and sector ─────────────────────────
# Grain: one row per (country, sector, cn_code).
# This is the most granular output table and the source for all aggregations.
# Sorted by CBAM cost descending so highest-exposure rows appear first.

df_cost_by_country_sector = (
    df_merged[[
        'country', 'iso2', 'sector', 'cn_code',
        'production_route_code', 'production_route',
        'import_tonnes', 'default_2026',
        'embedded_co2_tco2', 'cbam_cost_eur'
    ]]
    .copy()
    .sort_values('cbam_cost_eur', ascending=False)
    .reset_index(drop=True)
)

print(f'Output table 1 shape: {df_cost_by_country_sector.shape}')
df_cost_by_country_sector.head(10)

Output table 1 shape: (10641, 10)


,country,iso2,sector,cn_code,production_route_code,production_route,import_tonnes,default_2026,embedded_co2_tco2,cbam_cost_eur
0,Russia,RU,Iron and Steel,72071210,(C),"Carbon Steel, BF-BOF","3,152,317.10",3.53,"11,130,831.67","838,819,474.61"
1,India,IN,Iron and Steel,7208,(C),"Carbon Steel, BF-BOF","1,584,475.51",4.71,"7,459,710.71","562,163,798.79"
2,China,CN,Iron and Steel,7308,(C),"Carbon Steel, BF-BOF","959,471.44",6.64,"6,369,451.15","480,001,839.00"
3,Indonesia,ID,Iron and Steel,7208,(C),"Carbon Steel, BF-BOF","641,727.83",9.05,"5,809,562.09","437,808,599.12"
4,India,IN,Iron and Steel,7210,(C),"Carbon Steel, BF-BOF","1,007,722.43",4.71,"4,744,357.21","357,534,758.98"
5,Turkey,TR,Iron and Steel,7208,(C),"Carbon Steel, BF-BOF","1,413,742.60",2.67,"3,775,064.42","284,488,854.85"
6,China,CN,Iron and Steel,7210,(C),"Carbon Steel, BF-BOF","997,240.02",3.53,"3,515,769.69","264,948,403.61"
7,Russia,RU,Iron and Steel,7201,NaN,NaN,"1,029,970.84",3.34,"3,444,222.48","259,556,605.76"
8,South Korea,KR,Iron and Steel,7208,(C),"Carbon Steel, BF-BOF","1,457,477.61",2.33,"3,396,394.79","255,952,311.17"
9,Vietnam,VN,Iron and Steel,7210,(C),"Carbon Steel, BF-BOF","1,219,943.14",2.61,"3,180,391.78","239,674,324.27"


In [51]:
# ── Build output table 2: cost by country (all sectors aggregated) ────────────
# Grain: one row per country. All 119 CBAM countries are present.
# Countries with no 2024 trade flow data appear with zero cost.
# Sector-level costs are pivoted into separate columns for stacked bar charts.

# Total cost per country
df_cost_by_country = (
    df_cost_by_country_sector
    .groupby(['country', 'iso2'], as_index=False)
    .agg(
        total_import_tonnes = ('import_tonnes',     'sum'),
        total_embedded_co2  = ('embedded_co2_tco2', 'sum'),
        total_cbam_cost_eur = ('cbam_cost_eur',     'sum')
    )
    .sort_values('total_cbam_cost_eur', ascending=False)
    .reset_index(drop=True)
)

# Sector breakdown pivoted into columns and merged in
df_sector_pivot = (
    df_cost_by_country_sector
    .groupby(['country', 'sector'])['cbam_cost_eur']
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)
df_sector_pivot.columns = [
    f'cost_{c.lower().replace(" ", "_")}_eur' if c != 'country' else 'country'
    for c in df_sector_pivot.columns
]

df_cost_by_country = df_cost_by_country.merge(df_sector_pivot, on='country', how='left')

# Rank by total cost (1 = highest CBAM exposure)
df_cost_by_country.insert(
    0, 'rank',
    df_cost_by_country['total_cbam_cost_eur']
    .rank(ascending=False, method='min')
    .astype(int)
)

print(f'Output table 2 shape: {df_cost_by_country.shape}')
print(f'Countries with zero cost: {(df_cost_by_country["total_cbam_cost_eur"] == 0).sum()}')
df_cost_by_country.head(10)

Output table 2 shape: (119, 11)
Countries with zero cost: 21


,rank,country,iso2,total_import_tonnes,total_embedded_co2,total_cbam_cost_eur,cost_aluminium_eur,cost_cement_eur,cost_fertilizers_eur,cost_hydrogen_eur,cost_iron_and_steel_eur
0,1,China,CN,"8,095,975.83","36,573,116.70","2,756,150,074.87","308,151,498.35","5,475,440.14","58,292,961.74",15.46,"2,384,230,159.19"
1,2,Turkey,TR,"7,434,799.28","23,806,323.39","1,794,044,531.03","171,467,129.81","6,155,300.36","62,404,197.90",8.97,"1,554,017,893.98"
2,3,India,IN,"4,899,194.04","22,833,857.84","1,720,759,527.17","53,368,164.15","59,403.07","173,495.04",0.00,"1,667,158,464.91"
3,4,Russia,RU,"8,951,307.94","22,509,783.47","1,696,337,282.09","60,797,124.40",0.00,"319,152,347.10",0.00,"1,316,387,810.59"
4,5,Ukraine,UA,"11,546,697.32","12,614,285.90","950,612,585.28","939,133.19","188,129,221.28","3,529,407.10",0.00,"758,014,823.71"
5,6,Indonesia,ID,"1,106,398.01","9,736,165.35","733,717,421.05","3,238,281.63",0.00,"3,540.20",0.00,"730,475,599.22"
6,7,South Korea,KR,"3,634,304.40","9,473,206.16","713,900,816.34","8,731,297.34",348.20,"2,659,066.41",8.14,"702,510,096.24"
7,8,Vietnam,VN,"3,413,234.68","9,005,206.90","678,632,391.70","3,445,326.61",0.00,"34,570.45",0.00,"675,152,494.65"
8,9,United Kingdom,GB,"3,329,014.01","8,981,256.77","676,827,510.37","61,710,397.77","23,249,649.15","13,313,191.15","34,723.93","578,519,548.37"
9,10,Taiwan,TW,"2,605,715.25","7,872,095.20","593,241,094.43","494,836.90",413.67,"113,718.35",0.00,"592,632,125.51"


In [52]:
# ── Build output table 3: cost by sector (all countries aggregated) ───────────
# Grain: one row per sector.
# pct_of_total shows each sector's share of the total estimated CBAM bill.

df_cost_by_sector = (
    df_cost_by_country_sector
    .groupby('sector', as_index=False)
    .agg(
        n_countries         = ('country',           'nunique'),
        n_cn_codes          = ('cn_code',           'nunique'),
        total_import_tonnes = ('import_tonnes',     'sum'),
        total_embedded_co2  = ('embedded_co2_tco2', 'sum'),
        total_cbam_cost_eur = ('cbam_cost_eur',     'sum')
    )
    .sort_values('total_cbam_cost_eur', ascending=False)
    .reset_index(drop=True)
)

df_cost_by_sector['pct_of_total'] = (
    df_cost_by_sector['total_cbam_cost_eur']
    / df_cost_by_sector['total_cbam_cost_eur'].sum()
    * 100
).round(1)

print(f'Output table 3 shape: {df_cost_by_sector.shape}')
print(df_cost_by_sector.to_string(index=False))

Output table 3 shape: (5, 7)
        sector  n_countries  n_cn_codes  total_import_tonnes  total_embedded_co2  total_cbam_cost_eur  pct_of_total
Iron and Steel           47         200        67,716,428.14      165,554,602.29    12,476,194,828.29         80.20
     Aluminium           67          28         5,881,632.10       16,868,806.27     1,271,233,240.66          8.20
   Fertilizers           90          27        10,151,157.74       14,660,623.37     1,104,824,577.06          7.10
        Cement          100           6         6,734,810.49        9,244,657.10       696,677,359.06          4.50
      Hydrogen           93           1               117.62            1,938.99           146,122.04          0.00


In [53]:
# ── Sanity checks before writing to database ──────────────────────────────────
# Assertions and diagnostic prints to catch obvious problems before
# committing results to the database. Review all output carefully.

print('=== SANITY CHECKS ===')

# 1. All 119 CBAM countries present in output
n_countries = df_cost_by_country['country'].nunique()
assert n_countries == 119, f'Expected 119 countries, got {n_countries}'
print(f'PASS: All 119 CBAM countries present')

# 2. No negative costs
assert (df_cost_by_country_sector['cbam_cost_eur'] >= 0).all(), \
    'Negative CBAM costs found. Check for negative values in trade_flows.'
print('PASS: No negative costs')

# 3. No nulls in key output columns
key_cols = ['country', 'sector', 'import_tonnes', 'default_2026',
            'embedded_co2_tco2', 'cbam_cost_eur']
nulls = df_cost_by_country_sector[key_cols].isnull().sum()
if nulls.any():
    print(f'WARNING: Nulls found in key columns:')
    print(nulls[nulls > 0])
else:
    print('PASS: No nulls in key output columns')

# 4. All five expected sectors present
expected_sectors = {'Aluminium', 'Cement', 'Fertilizers', 'Hydrogen', 'Iron and Steel'}
actual_sectors   = set(df_cost_by_country_sector['sector'].unique())
missing          = expected_sectors - actual_sectors
if missing:
    print(f'WARNING: Expected sectors missing: {missing}')
else:
    print('PASS: All five sectors present')

# 5. Countries with zero cost — list for awareness, not an error
zero_cost = df_cost_by_country[df_cost_by_country['total_cbam_cost_eur'] == 0]['country'].tolist()
print(f'\nCountries with zero CBAM cost (no 2024 trade flow match): {len(zero_cost)}')
print(zero_cost)

# 6. Top 10 countries by cost — manual sense check
print('\nTop 10 countries by estimated CBAM cost:')
print(
    df_cost_by_country[['rank', 'country', 'total_cbam_cost_eur', 'total_embedded_co2']]
    .head(10)
    .to_string(index=False)
)

=== SANITY CHECKS ===
PASS: All 119 CBAM countries present
PASS: No negative costs
PASS: No nulls in key output columns
PASS: All five sectors present

Countries with zero CBAM cost (no 2024 trade flow match): 21
['Curacao', 'Brunei Darussalam', 'Congo', 'Cambodia', 'Eritrea', 'Yemen', 'Equatorial Guinea', 'Rwanda', 'Eswatini', 'Haiti', 'Papua New Guinea', 'Jamaica', 'Laos', 'Suriname', 'Sudan', 'Mongolia', 'Namibia', 'Sierra Leone', 'Nepal', 'North Korea', 'Mali']

Top 10 countries by estimated CBAM cost:
 rank        country  total_cbam_cost_eur  total_embedded_co2
    1          China     2,756,150,074.87       36,573,116.70
    2         Turkey     1,794,044,531.03       23,806,323.39
    3          India     1,720,759,527.17       22,833,857.84
    4         Russia     1,696,337,282.09       22,509,783.47
    5        Ukraine       950,612,585.28       12,614,285.90
    6      Indonesia       733,717,421.05        9,736,165.35
    7    South Korea       713,900,816.34        9,473

In [54]:
# ── Write output tables to database ──────────────────────────────────────────
# Write all three output tables to cbam.db.
# if_exists='replace' makes this notebook fully idempotent and safe to re-run.

tables = {
    'cbam_cost_by_country_sector': df_cost_by_country_sector,
    'cbam_cost_by_country':        df_cost_by_country,
    'cbam_cost_by_sector':         df_cost_by_sector,
}

for table_name, df in tables.items():
    df.to_sql(table_name, con, if_exists='replace', index=False)
    print(f'Written: {table_name} ({len(df):,} rows)')

print('\nAll output tables written successfully.')

Written: cbam_cost_by_country_sector (10,641 rows)
Written: cbam_cost_by_country (119 rows)
Written: cbam_cost_by_sector (5 rows)

All output tables written successfully.


In [55]:
# ── Verification queries ──────────────────────────────────────────────────────
# Confirm all three tables exist in the database with the correct row counts.

print('=== DATABASE VERIFICATION ===')
for table_name, df in tables.items():
    count    = pd.read_sql(f'SELECT COUNT(*) AS n FROM {table_name}', con).iloc[0, 0]
    expected = len(df)
    status   = 'PASS' if count == expected else 'FAIL'
    print(f'[{status}] {table_name}: {count:,} rows (expected {expected:,})')

con.close()
print('\nConnection closed. Notebook complete.')

=== DATABASE VERIFICATION ===
[PASS] cbam_cost_by_country_sector: 10,641 rows (expected 10,641)
[PASS] cbam_cost_by_country: 119 rows (expected 119)
[PASS] cbam_cost_by_sector: 5 rows (expected 5)

Connection closed. Notebook complete.
